In [1]:
import pandas as pd
import numpy as np

In [2]:
samples_df = pd.read_csv('MOMI_derived_data_Oct4_mapping.csv')

/tmp/ipykernel_1848/2479512848.py:1: DtypeWarning: Columns (0,4,7,9,11,61,81,84,87,90,92,108) have mixed types. Specify dtype option on import or set low_memory=False.
  samples_df = pd.read_csv('MOMI_derived_data_Oct4_mapping.csv')


In [3]:
filtered_df = samples_df[(samples_df['SITE']=='AMANHIT')]

In [24]:
filtered_df.head()

,ORIG_ID,SAMPLE_ID,SAMPLE_TYPE,VISITDT,DEL_DATE_x,SITE,ORIG_ID+VISITDT,ORIG_ID+DEL_DATE,SINGLE_TWIN_x,BABY_ID_x,...,ERLY_SB,LT_SB,SB_NEW,PE_PRIORITY,PE_NEW,PE_CAT,GA_weeks,Trimester,GAGEBRTH_weeks,GFR
9857,16580,20BE0000901,Maternal Plasma,2014-06-09,NaN,AMANHIT,16580_41799,NaN,1.0,NaN,...,NaN,NaN,0.0,NaN,0.0,NaN,19.0,second,39.0,NaN
9858,16580,20BA0005301,Maternal Plasma,2014-09-25,NaN,AMANHIT,16580_41907,NaN,1.0,NaN,...,NaN,NaN,0.0,NaN,0.0,NaN,34.0,third,39.0,NaN
9859,16683,20BE0000201,Maternal Plasma,2014-06-06,NaN,AMANHIT,16683_41796,NaN,1.0,NaN,...,NaN,NaN,0.0,NaN,0.0,NaN,9.0,first,39.0,NaN
9860,16683,20BA0006501,Maternal Plasma,2014-10-10,NaN,AMANHIT,16683_41922,NaN,1.0,NaN,...,NaN,NaN,0.0,NaN,0.0,NaN,27.0,third,39.0,109.735933
9861,16685,20BE0005506,Maternal Plasma,2014-06-23,NaN,AMANHIT,16685_41813,NaN,1.0,NaN,...,NaN,NaN,0.0,NaN,0.0,NaN,10.0,first,42.0,NaN


In [4]:
filtered_df['GAGEBRTH_weeks']=(filtered_df['GAGEBRTH_NEW']+6)//7

/tmp/ipykernel_1848/527809794.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['GAGEBRTH_weeks']=(filtered_df['GAGEBRTH_NEW']+6)//7


In [5]:
def calculate_gfr(serum_creatinine, age, female=True, black=True):
    """
    Calculates the estimated Glomerular Filtration Rate (eGFR) using the CKD-EPI formula,
    adjusted for population characteristics. This function estimates kidney function
    based on serum creatinine levels, age, and gender, and is specifically adjusted
    for the Tanzanian population with considerations for gender and assumed racial factors.

    Parameters:
    serum_creatinine (float): The patient's serum creatinine level (mg/dL).
    age (int): The patient's age in years.
    female (bool): True if the patient is female, False if male. Defaults to True.

    Returns:
    float: The estimated GFR value adjusted for the Tanzanian population.

    Notes:
    - The constants for the CKD-EPI equation are set based on the gender.
    - Female and black is the default setting for the adjustment factor.
    - It's important to ensure correct input as the function does not validate the values.
    """
    # Constants for the CKD-EPI equation
    kappa_female = 0.7  # κ value for females
    kappa_male = 0.9  # κ value for males
    alpha_female = -0.329  # α value for females
    alpha_male = -0.411  # α value for males
    gfr_female_constant = 1.018  # Constant factor if the patient is female
    gfr_black_constant = 1.159  # Constant factor if the patient is black

    # Set kappa and alpha based on gender
    kappa = kappa_female if female else kappa_male
    alpha = alpha_female if female else alpha_male

    # Calculate the GFR using the CKD-EPI equation
    scr_kappa_ratio = serum_creatinine / kappa
    min_value = np.minimum(scr_kappa_ratio, 1) ** alpha
    max_value = np.maximum(scr_kappa_ratio, 1) ** -1.209
    age_factor = 0.993**age

    gfr = 141 * min_value * max_value * age_factor

    # Adjust GFR for gender and race
    if female:
        gfr *= gfr_female_constant
    # If the population is black; adjust the GFR
    if black:
        gfr *= gfr_black_constant

    return gfr

In [6]:
filtered_df['GFR'] = calculate_gfr(filtered_df['SCR'], filtered_df['PW_AGE'], female=True)

/tmp/ipykernel_1848/496771750.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['GFR'] = calculate_gfr(filtered_df['SCR'], filtered_df['PW_AGE'], female=True)


In [7]:
import pandas as pd

# --- common filter ---
common_filter = (
    "(SINGLE_TWIN_x == 1) and "
    "~GAGEBRTH_NEW.isna() and "
    "PTB_NEW == 0 and "
    "PE_NEW == 0 and "
    "SGA_10 == 0 and "
    "SB_NEW == 0"
)

# --- cohort filter ---
ptb_cohort_filter = "(SPONT_LABOUR != 2)"

# --- control filter ---
ptb_control_filter = (
    "(GAGEBRTH_weeks >= 39) and "
    "(GAGEBRTH_weeks < 41) and "
    "(BIRTH_WEIGHT >= 2500) and "
    "(CHRON_HTN != 1) and "
    "((GFR >= 60) or GFR.isna()) and "
    "(DIABETES != 1) and "
    "(FETAL_ANOMALIES != 1) and "
    "(CONG_ANOMALIES != 1) and "
    "(HIV != 1) and "
    "(MALARIA != 1) and "
    "(SYPHILIS != 1)"
)

# --- control group ---
control_ids = filtered_df.query(f"{common_filter} and {ptb_control_filter}")["SAMPLE_ID"]

# --- cohort group ---
cohort_ids = filtered_df.query(f"{ptb_cohort_filter} and (PTB_NEW == 1)")["SAMPLE_ID"]

# --- merge ---
final_ids = pd.concat([control_ids, cohort_ids]).unique()


In [8]:
len(final_ids)

3515

In [9]:
filtered_df[filtered_df['SAMPLE_ID'].isin(final_ids)]['PTB_NEW'].value_counts()

0.0    3202
1.0     313
Name: PTB_NEW, dtype: int64

In [10]:
T1_ids = filtered_df[filtered_df['SAMPLE_ID'].isin(final_ids)]['SAMPLE_ID'].unique()

In [11]:
len(T1_ids)

3515

In [12]:
features_df = pd.read_csv('maternal_feature_intensities.csv')

In [13]:
features_df = features_df.T.reset_index()

In [14]:
features_df.columns = features_df.iloc[0]
features_df = features_df.drop([0]).reset_index(drop=True)
features_df = features_df.rename(columns = {'feature_label':'SAMPLE_ID'})

In [15]:
features_df.head()

,SAMPLE_ID,maternal_plasma_1m0002006i00,maternal_plasma_1m0002108i00,maternal_plasma_1m0002208i00,maternal_plasma_1m0002208i01,maternal_plasma_1m0002306i00,maternal_plasma_1m0002308i00,maternal_plasma_1m0002308i01,maternal_plasma_1m0002402i00,maternal_plasma_1m0002406i00,...,maternal_plasma_2m0162808i00,maternal_plasma_2m0162820i00,maternal_plasma_2m0162921i00,maternal_plasma_2m0164377i00,maternal_plasma_2m0164477i00,maternal_plasma_2targeted_peak_1,maternal_plasma_2targeted_peak_11,maternal_plasma_2targeted_peak_3,maternal_plasma_2targeted_peak_6,maternal_plasma_2targeted_peak_7
0,20BA0000101,787.0,208.0,7740.0,5011.0,382.0,480.0,388.0,230.0,979.0,...,NaN,235.0,203.0,137.0,148.0,396164.0,131053.0,32175.0,100396.0,148613.0
1,20BA0000201,705.0,317.0,7910.0,4840.0,308.0,574.0,384.0,252.0,841.0,...,NaN,241.0,144.0,164.0,NaN,375017.0,122484.0,24771.0,87466.0,126241.0
2,20BA0000302,1064.0,205.0,7462.0,4508.0,370.0,560.0,433.0,138.0,1097.0,...,NaN,148.0,138.0,133.0,112.0,360652.0,124519.0,24528.0,86525.0,133546.0
3,20BA0000401,649.0,196.0,11322.0,5738.0,244.0,800.0,419.0,304.0,643.0,...,NaN,NaN,166.0,144.0,160.0,414069.0,128132.0,23785.0,89241.0,125161.0
4,20BA0000501,818.0,173.0,6946.0,4082.0,262.0,351.0,374.0,121.0,926.0,...,NaN,185.0,166.0,130.0,110.0,445008.0,122516.0,29179.0,93706.0,131963.0


In [26]:
labels_df = filtered_df[(filtered_df['SAMPLE_ID']).isin(T1_ids)]

In [27]:
labels_df.head()

,ORIG_ID,SAMPLE_ID,SAMPLE_TYPE,VISITDT,DEL_DATE_x,SITE,ORIG_ID+VISITDT,ORIG_ID+DEL_DATE,SINGLE_TWIN_x,BABY_ID_x,...,ERLY_SB,LT_SB,SB_NEW,PE_PRIORITY,PE_NEW,PE_CAT,GA_weeks,Trimester,GAGEBRTH_weeks,GFR
9857,16580,20BE0000901,Maternal Plasma,2014-06-09,NaN,AMANHIT,16580_41799,NaN,1.0,NaN,...,NaN,NaN,0.0,NaN,0.0,NaN,19.0,second,39.0,NaN
9858,16580,20BA0005301,Maternal Plasma,2014-09-25,NaN,AMANHIT,16580_41907,NaN,1.0,NaN,...,NaN,NaN,0.0,NaN,0.0,NaN,34.0,third,39.0,NaN
9859,16683,20BE0000201,Maternal Plasma,2014-06-06,NaN,AMANHIT,16683_41796,NaN,1.0,NaN,...,NaN,NaN,0.0,NaN,0.0,NaN,9.0,first,39.0,NaN
9860,16683,20BA0006501,Maternal Plasma,2014-10-10,NaN,AMANHIT,16683_41922,NaN,1.0,NaN,...,NaN,NaN,0.0,NaN,0.0,NaN,27.0,third,39.0,109.735933
9862,16687,20BE0000601,Maternal Plasma,2014-06-07,NaN,AMANHIT,16687_41797,NaN,1.0,NaN,...,NaN,NaN,0.0,NaN,0.0,NaN,14.0,second,40.0,NaN


In [28]:
labels_df = labels_df[['SAMPLE_ID', 'ORIG_ID', 'PTB_NEW', 'Trimester']]

In [29]:
labels_df.shape

(3515, 4)

In [ ]:
data = pd.merge(features_df, labels_df, on='SAMPLE_ID', how='inner')

In [ ]:
data.shape

(3515, 45702)

In [ ]:
data.head()

,SAMPLE_ID,maternal_plasma_1m0002006i00,maternal_plasma_1m0002108i00,maternal_plasma_1m0002208i00,maternal_plasma_1m0002208i01,maternal_plasma_1m0002306i00,maternal_plasma_1m0002308i00,maternal_plasma_1m0002308i01,maternal_plasma_1m0002402i00,maternal_plasma_1m0002406i00,...,maternal_plasma_2m0164377i00,maternal_plasma_2m0164477i00,maternal_plasma_2targeted_peak_1,maternal_plasma_2targeted_peak_11,maternal_plasma_2targeted_peak_3,maternal_plasma_2targeted_peak_6,maternal_plasma_2targeted_peak_7,ORIG_ID,PTB_NEW,Trimester
0,20BA0000501,818.0,173.0,6946.0,4082.0,262.0,351.0,374.0,121.0,926.0,...,130.0,110.0,445008.0,122516.0,29179.0,93706.0,131963.0,16750,0.0,second
1,20BA0000801,591.0,165.0,6421.0,3499.0,306.0,340.0,231.0,180.0,737.0,...,170.0,175.0,436401.0,125233.0,21584.0,91637.0,136345.0,17294,0.0,second
2,20BA0000901,450.0,202.0,3751.0,2178.0,311.0,298.0,155.0,128.0,560.0,...,NaN,107.0,386312.0,123776.0,20451.0,83933.0,122383.0,17350,0.0,second
3,20BA0001101,364.0,153.0,11665.0,6499.0,341.0,664.0,393.0,172.0,838.0,...,187.0,110.0,418080.0,125097.0,26388.0,88640.0,137988.0,16687,0.0,second
4,20BA0001402,821.0,168.0,6448.0,3191.0,330.0,426.0,340.0,NaN,894.0,...,NaN,180.0,417227.0,118967.0,24056.0,86882.0,123378.0,16731,1.0,second


In [ ]:
data.to_csv('All_Trimesters.csv')